# HHL and the fine print

Solving `Ax = b` on a quantum computer, verifying it is exact, and then measuring
every caveat that decides whether the exponential speedup survives contact with a
real problem.

Companion to [`tutorials/03-pdes/01-hhl-linear-systems-intro.md`](../tutorials/03-pdes/01-hhl-linear-systems-intro.md).

**Requires:** `pip install -e ".[qiskit]"`

## 1. The system

HHL's accuracy depends on the eigenvalues landing exactly on clock-register
values, so the matrix is built **eigenvalue-first**. A matrix chosen for looks
rather than spectrum makes the algorithm appear broken when it is only
mis-encoded.

In [1]:
import numpy as np
from qprac_lab.algorithms.pdes.hhl_intro import well_conditioned_system

matrix, rhs = well_conditioned_system()
eigenvalues = np.linalg.eigvalsh(matrix)

print("A =", matrix.tolist())
print("b =", rhs.tolist())
print(f"eigenvalues: {eigenvalues}   condition number kappa = {eigenvalues.max()/eigenvalues.min():.1f}")
print("classical solution:", np.linalg.solve(matrix, rhs))

A = [[1.5, 0.5], [0.5, 1.5]]
b = [1.0, 0.0]
eigenvalues: [1. 2.]   condition number kappa = 2.0
classical solution: [ 0.75 -0.25]


## 2. Encoding the eigenvalues

Phase estimation resolves `lambda * t / (2*pi)` as an `n`-bit fraction, so
`lambda * t / (2*pi) * 2**n` must be an integer in `1 .. 2**n - 1`.

Two ways this goes wrong: too large a `t` and the biggest eigenvalue wraps to
phase zero, silently deleting its part of the solution; a `t` that lands
off-integer and phase estimation smears that eigenvalue across neighbouring
registers.

In [2]:
from qprac_lab.algorithms.pdes.hhl_intro import (
    clock_values, eigenvalues_representable, suggested_evolution_time,
)

chosen = suggested_evolution_time(eigenvalues, 2)
print(f"chosen t = {chosen:.6f}")
print(f"clock values = {clock_values(eigenvalues, chosen, 2)}  (must be integers in 1..3)")
print(f"exactly representable: {eigenvalues_representable(eigenvalues, chosen, 2)}")

print(f"\nt = pi aliases lambda=2 onto clock value 4 == 0 (mod 4):")
print(f"  clock values = {clock_values(eigenvalues, np.pi, 2)}")
print(f"  representable: {eigenvalues_representable(eigenvalues, np.pi, 2)}")

chosen t = 1.570796
clock values = [1.0, 2.0]  (must be integers in 1..3)
exactly representable: True

t = pi aliases lambda=2 onto clock value 4 == 0 (mod 4):
  clock values = [2.0, 4.0]
  representable: False


## 3. Solve

QPE puts the eigenvalues in a clock register, a controlled rotation writes
`1/lambda` into an ancilla amplitude, inverse QPE uncomputes the clock, and
postselecting the ancilla on 1 leaves `|A^-1 b>`.

In [3]:
from qprac_lab.algorithms.pdes.hhl_intro import solve_hhl

solution, success, circuit, evolution_time = solve_hhl(matrix, rhs, num_clock_qubits=2)
exact = np.linalg.solve(matrix, rhs)
exact_normalised = exact / np.linalg.norm(exact)

print(f"HHL       : {np.round(np.real(solution), 9)}")
print(f"classical : {np.round(exact_normalised, 9)}")
print(f"fidelity  : {abs(np.vdot(solution, exact_normalised))**2:.12f}")
print(f"\nP(ancilla = 1) = {success:.4f}   <- the rest of the runs are discarded")
print(f"circuit depth  = {circuit.decompose(reps=4).depth()}")

HHL       : [ 0.9486833  -0.31622777]
classical : [ 0.9486833  -0.31622777]
fidelity  : 1.000000000000

P(ancilla = 1) = 0.6250   <- the rest of the runs are discarded


circuit depth  = 86


Exact to twelve digits. Now the three things that decide whether that matters.

## 4. Caveat one: the encoding has to be right

Varying only the evolution time, changing nothing else:

In [4]:
from qprac_lab.algorithms.pdes.hhl_intro import run_hhl_intro_tutorial

result = run_hhl_intro_tutorial()

print(f"{'t':>8} {'clock values':>22} {'exact?':>7} {'fidelity':>11}")
for row in result.encoding_study:
    print(f"{row['evolution_time']:>8.4f} {str([round(c, 3) for c in row['clock_values']]):>22} "
          f"{str(row['exactly_representable']):>7} {row['fidelity']:>11.6f}")

       t           clock values  exact?    fidelity
  1.5708             [1.0, 2.0]    True    1.000000
  0.7854             [0.5, 1.0]   False    0.630555
  0.5236         [0.333, 0.667]   False    0.480871
  0.3927            [0.25, 0.5]   False    0.443705


One parameter, and fidelity falls from 1.0 to 0.44.

In practice you do not know the spectrum in advance — computing it is the sort of
thing you wanted the quantum computer for — so choosing `t` well is
chicken-and-egg. And the general case is *worse* than these rows: a real spectrum
is incommensurate, so **no** `t` makes every eigenvalue exact.

## 5. Caveat two: conditioning

HHL's runtime carries `kappa^2`. That shows up as discarded shots.

In [5]:
print(f"{'kappa':>7} {'eigenvalues':>16} {'fidelity':>11} {'P(success)':>11}")
for row in result.conditioning_study:
    print(f"{row['condition_number']:>7.1f} {str([round(v, 2) for v in row['eigenvalues']]):>16} "
          f"{row['fidelity']:>11.6f} {row['success_probability']:>11.4f}")

  kappa      eigenvalues    fidelity  P(success)
    2.0       [1.0, 2.0]    1.000000      0.6250
    4.0       [1.0, 4.0]    0.983968      0.4206
    8.0       [1.0, 8.0]    0.558323      0.0982


At `kappa = 8`, barely a tenth of runs survive postselection **and** the surviving
answer has lost half its fidelity.

## 6. Caveat three: readout, which dissolves the speedup

HHL returns `|x>`, a state whose amplitudes encode the solution. It does not
return `x`. Recovering all `N` amplitudes to precision `eps` costs `O(N / eps^2)`
measurements.

In [6]:
from qprac_lab.algorithms.pdes.hhl_intro import measurements_for_precision

print(f"{'precision':>10} {'shots for N=2':>14} {'shots for N=1024':>18}")
for precision in (0.1, 0.01, 0.001):
    print(f"{precision:>10.3f} {measurements_for_precision(2, precision):>14,} "
          f"{measurements_for_precision(1024, precision):>18,}")

print(f"\n{result.caveats['shots_to_read_full_solution']:,} shots to read a TWO-element "
      f"solution to 1% -- for a system numpy solves exactly and instantly.")
print(f"scaling: {result.caveats['readout_scaling']}")

 precision  shots for N=2   shots for N=1024
     0.100            199            102,399
     0.010         20,000         10,240,000
     0.001      2,000,000      1,024,000,000

20,000 shots to read a TWO-element solution to 1% -- for a system numpy solves exactly and instantly.
scaling: O(N / precision^2) -- linear in N, which is the speedup


`O(N)` — **linear in the size the algorithm was supposed to be logarithmic in.**

If you read out the solution vector, you have thrown the speedup away. HHL is only
interesting when you want a single summary number like `<x|M|x>`, and never `x`.

## When not to use this

- **When you want the solution vector.** Readout is `O(N)`. This is the big one.
- **When `A` is ill-conditioned.** 90% of runs discarded at `kappa = 8`.
- **When `|b>` is not cheaply preparable.** Loading a general vector is `O(N)`,
  which erases the advantage before the algorithm starts.
- **When you do not know the spectrum**, which is the normal case.
- **On current hardware.** Depth 86 for a 2x2 system.

The realistic framing is HHL as a **subroutine** inside a larger quantum algorithm
where its output stays quantum. As a standalone linear solver it is not
competitive and not close. See
[the variational alternative](06-variational-heat-equation.ipynb) for shallower
circuits at the cost of a non-convex optimisation.